In [1]:
from pathlib import Path
import json
import shutil

from src.application.settings import ViRAGESettings
from src.domain.enums import ChartCaseType
from src.domain.models import DataProfile, QueryUnderstandingResult
from src.infrastructure.runtime import RuntimeContext
from src.services.visrag import VisRAGService
from rich import print as rprint

In [2]:
# Рабочая папка для демо
demo_root = Path("./notebook_visrag_demo").resolve()
if demo_root.exists():
    shutil.rmtree(demo_root)

corpus_root = demo_root / "corpora"
artifact_root = demo_root / "artifacts"
corpus_root.mkdir(parents=True, exist_ok=True)
artifact_root.mkdir(parents=True, exist_ok=True)

# Мини-корпус Plot2Code в нормализованном формате
examples = [
    {
        "id": "plot2code-line-001",
        "corpus": "plot2code",
        "source": "Plot2Code",
        "chart_type": "line",
        "instruction": "Show the sales trend over time by month.",
        "description": "A line chart for monthly sales over time.",
        "tags": ["trend analysis", "time series", "business"],
        "code_language": "python",
        "domain": "business",
        "complexity": "canonical",
    },
    {
        "id": "plot2code-bar-001",
        "corpus": "plot2code",
        "source": "Plot2Code",
        "chart_type": "bar",
        "instruction": "Compare average sales across regions.",
        "description": "A bar chart for grouped comparison across categories.",
        "tags": ["comparison", "categories"],
        "code_language": "python",
        "domain": "business",
        "complexity": "canonical",
    },
    {
        "id": "plot2code-scatter-001",
        "corpus": "plot2code",
        "source": "Plot2Code",
        "chart_type": "scatter",
        "instruction": "Show the relationship between profit and sales.",
        "description": "A scatter plot for correlation analysis.",
        "tags": ["relationship analysis", "correlation"],
        "code_language": "python",
        "domain": "business",
        "complexity": "canonical",
    },
]

with (corpus_root / "plot2code.jsonl").open("w", encoding="utf-8") as f:
    for row in examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

settings = ViRAGESettings(
    artifact_root=artifact_root,
    visrag_corpus_root=corpus_root,
    visrag_rag_enabled=True,
    visrag_retriever_backend="local_tfidf",   # для ноутбука без Ollama
    visrag_retriever_top_k=3,
    visrag_retriever_fetch_k=10,
    visrag_enable_llm_refinement=False,
)

runtime = RuntimeContext(settings=settings, llm=None)

query_understanding = QueryUnderstandingResult(
    intent="Покажи тренд продаж по датам",
    requested_operations=["trend analysis"],
    candidate_charts=["line", "bar"],
    constraints=[],
    case_type=ChartCaseType.CANONICAL,
    confidence=0.92,
)

data_profile = DataProfile(
    row_count=120,
    col_count=4,
    columns=[],
    likely_numeric_columns=["sales"],
    likely_categorical_columns=["region"],
    likely_time_columns=["date"],
    quality_notes=["Detected time-like columns: date."],
)

result = VisRAGService().invoke(
    query_understanding=query_understanding,
    data_profile=data_profile,
    runtime=runtime,
    planning=None,   # если в твоей версии planning обязателен, подставь объект plan
)

rprint(result)

AttributeError: 'NoneType' object has no attribute 'mode'

In [ ]:
rprint("=== RETRIEVAL STRATEGY ===")
rprint(result.retrieval_strategy)

rprint("\n=== CORPUS STATUS ===")
rprint(result.corpus_status)

rprint("\n=== TOP RECOMMENDATIONS ===")
for rec in result.recommendations:
    rprint({
        "priority": rec.priority,
        "chart_family": rec.chart_family,
        "score": getattr(rec, "score", None),
        "support_examples": getattr(rec, "support_examples", []),
        "rationale": rec.rationale,
    })

rprint("\n=== VISUALIZATION PLAN ===")
plan = result.visualization_plan
rprint({
    "chart_family": plan.chart_family,
    "visual_task": plan.visual_task,
    "goal": plan.goal,
    "title": plan.title,
    "confidence": plan.confidence,
})

rprint("\nfield_bindings:")
for binding in plan.field_bindings:
    rprint(binding.model_dump())

rprint("\naxes:")
for axis in plan.axes:
    rprint(axis.model_dump())

rprint("\nbuild_instructions:")
for item in plan.build_instructions:
    rprint("-", item)

rprint("\n=== RETRIEVED EXAMPLES ===")
for ex in result.retrieved_examples:
    rprint({
        "example_id": ex.example_id,
        "source": ex.source,
        "chart_type": ex.chart_type,
        "score": getattr(ex, "score", None),
        "instruction": ex.instruction,
        "rationale": ex.rationale,
    })

rprint("\n=== FULL RESULT JSON ===")
rprint(result.model_dump_json(indent=2, ensure_ascii=False))